# 02. Схема валидации и baseline

**Валидация** (`src/validation.py`) повторяет ситуацию теста на прошлых датах. На 15.12.2025 впереди
17 матчей, это весь остаток сезона, и на 6 из них продажи уже открыты. Для каждой даты «псевдо-сегодня» `as_of`:
- **train**: матчи, сыгранные до `as_of`, из `build_table(as_of)`;
- **val**: следующие до 17 матчей того же сезона. Признаки берутся из `build_table(as_of)`, итог из `build_table()`;
- прогноз обрезается по правилам сдачи: `sold_as_of ≤ ŷ ≤ capacity`, целое.

**Baseline**: среднее `tickets_total` по зоне на train фолда.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
from IPython.display import Markdown, display

from src.data import build_table
from src.validation import FOLDS, evaluate, make_fold, predict_test

pd.set_option("display.precision", 1)

## Фолды

In [2]:
full = build_table()
rows = []
for as_of in FOLDS:
    train, val, y = make_fold(as_of, full)
    ids = val["match_id"].unique()
    last = train["match_date"].max()
    rows.append({
        "as_of": as_of,
        "train_matches": train["match_id"].nunique(),
        "val": f"{ids[0]}–{ids[-1]}",
        "val_matches": len(ids),
        "val_in_sale": val.groupby("match_id")["sales_started"].max().sum(),
        "days_since_last_game": (pd.Timestamp(as_of) - last).days,
    })
test = full[full["is_test"] == 1]
rows.append({"as_of": "2025-12-15 (test)", "train_matches": full["tickets_total"].notna().sum() // 8,
             "val": "M086–M102", "val_matches": test["match_id"].nunique(),
             "val_in_sale": test.groupby("match_id")["sales_started"].max().sum(),
             "days_since_last_game": (pd.Timestamp("2025-12-15") - full.loc[full["is_test"] == 0, "match_date"].max()).days})
pd.DataFrame(rows).set_index("as_of")

,train_matches,val,val_matches,val_in_sale,days_since_last_game
as_of,,,,,
2023-12-15,20,M021–M034,14,5,10
2024-11-25,51,M052–M068,17,5,3
2025-09-20,73,M074–M085,12,3,3
2025-12-15 (test),85,M086–M102,17,6,8


## Baseline: среднее по зоне

In [3]:
def baseline_predict(train, val, as_of):
    zone_mean = train.groupby("zone")["tickets_total"].mean()
    return val["zone"].map(zone_mean).to_numpy()


res = evaluate(baseline_predict)
res

EvalResult(RMSE=141.04, folds=['2023-12-15', '2024-11-25', '2025-09-20'])

### RMSE по фолдам

In [4]:
res.by_fold

,rmse,matches,rows
fold,,,
2023-12-15,172.2,14,112
2024-11-25,103.0,17,136
2025-09-20,147.2,12,96
all,141.0,43,344


### RMSE по зонам

In [5]:
res.by_zone

fold,2023-12-15,2024-11-25,2025-09-20,all
zone,,,,
1,46.4,45.4,27.8,41.6
2,40.4,28.5,47.1,38.4
3,63.6,57.0,71.0,63.3
4,220.1,117.2,144.2,164.3
5,130.1,110.0,77.6,109.4
6,286.4,204.3,286.1,257.0
7,282.8,86.4,227.8,208.5
8,44.7,61.6,68.3,58.8


### RMSE: матчи, уже открытые к продаже на as_of, и ещё не открытые

In [6]:
res.by_status

,rmse,matches,rows
in_sale,,,
ещё не в продаже,149.5,30,240
в продаже,119.3,13,104


In [7]:
display(Markdown(f"# Baseline RMSE = {res.overall:.1f}"))

# Baseline RMSE = 141.0

## Где baseline ошибается

In [8]:
p = res.preds.assign(err=lambda d: d["y_pred"] - d["y_true"])
by_match = (p.groupby(["fold", "match_id"])
              .agg(y_true=("y_true", "sum"), y_pred=("y_pred", "sum"), in_sale=("in_sale", "max"))
              .assign(err=lambda d: d["y_pred"] - d["y_true"])
              .merge(full.drop_duplicates("match_id").set_index("match_id")[["match_date", "opponent", "weekday"]],
                     left_on="match_id", right_index=True))
by_match.reindex(by_match["err"].abs().sort_values(ascending=False).index).head(10)

y_true  y_pred  in_sale   err match_date      opponent  \
fold       match_id                                                           
2025-09-20 M079        5691    4608    False -1083 2025-10-27  Металлург Мг   
2023-12-15 M022        3802    4708     True   906 2023-12-21          Сочи   
2025-09-20 M075        5649    4757     True  -892 2025-10-03     Локомотив   
           M076        5449    4608     True  -841 2025-10-06  Автомобилист   
2024-11-25 M053        3846    4642     True   796 2024-12-04          Сочи   
2023-12-15 M034        5301    4708    False  -593 2024-02-22          Лада   
2025-09-20 M084        4033    4608    False   575 2025-12-04          Сочи   
2023-12-15 M021        4252    4827     True   575 2023-12-19       Ак Барс   
2025-09-20 M080        5165    4608    False  -557 2025-10-29         Барыс   
           M081        5140    4608    False  -532 2025-11-01    Нефтехимик   

                     weekday  
fold       match_id           
2025-09-20 M079            1  
2023-12-15 M022            4  
2025-09-20 M075            5  
           M076            1  
2024-11-25 M053            3  
2023-12-15 M034            4  
2025-09-20 M084            4  
2023-12-15 M021            2  
2025-09-20 M080            3  
           M081            6

In [9]:
# Средняя ошибка (bias) по зонам: > 0 — baseline завышает
p.pivot_table(index="zone", columns="fold", values="err", aggfunc="mean")

fold,2023-12-15,2024-11-25,2025-09-20
zone,,,
1,25.9,-33.6,4.8
2,20.7,24.3,43.8
3,34.6,15.6,44.5
4,-150.8,-103.4,-139.1
5,-84.5,-29.6,-63.3
6,-92.7,200.2,-210.6
7,118.4,-41.4,-129.6
8,-25.4,-35.8,-2.0


**Выводы по ошибкам baseline:**
- Почти вся ошибка приходится на массовые зоны 4, 6 и 7: RMSE 160–260 против 40–65 в зонах 1–3 и 8.
- Смещение по зонам меняет знак от фолда к фолду. В зоне 6 baseline завышает на 200 в 2024/25 и занижает на 210 в 2025/26. Уровень спроса в зоне плывёт между сезонами, а общее среднее этого не видит. Стоит проверить, как меняются `capacity`/`season_tickets` и насколько полезно взвешивать недавние матчи.
- В сезоне 2025/26 baseline системно занижает: M075, M076, M079–M081 недопрогнозированы на 500–1100 билетов за матч. Спрос текущего сезона выше исторического среднего.
- Baseline одинаков для всех матчей. Различия по сопернику, дню недели и дате не учитываются, отсюда ошибки ±600–900 на отдельных матчах (M022, M053).
- По матчам, уже открытым к продаже, RMSE ниже (119 против 149) отчасти за счёт нижней границы `sold_as_of`. Темп уже начавшихся продаж должен стать сильным признаком.
- В первом фолде (2023-12-15) train всего 20 матчей, поэтому его RMSE заметно шумнее остальных.

## Прогноз на test

In [10]:
sub = predict_test(baseline_predict)
assert len(sub) == 136 and (sub["tickets_total"] >= 0).all()
assert sub["tickets_total"].dtype.kind == "i"

out = Path("../submissions")
out.mkdir(exist_ok=True)
sub.to_csv(out / "baseline.csv", index=False, encoding="utf-8")
sub.pivot(index="match_id", columns="zone", values="tickets_total")

zone,1,2,3,4,5,6,7,8
match_id,,,,,,,,
M086,465,308,633,782,767,487,781,450
M087,465,308,633,782,767,700,781,450
M088,465,308,633,782,767,518,781,450
M089,465,308,633,782,767,621,781,450
M090,465,308,633,782,767,487,781,450
M091,465,308,633,782,767,487,781,450
M092,465,308,633,782,767,487,781,450
M093,465,308,633,782,767,487,781,450
M094,465,308,633,782,767,487,781,450


# Умный baseline

Выше — baseline (среднее по зоне за все прошлые матчи, **RMSE 141.0**). Здесь две доработки:

**Уровень зоны** — среднее `tickets_total` по зоне за недавние матчи до `as_of`. Окно двух видов:
- `last` — последние N сыгранных матчей, окно может переходить через границу сезонов;
- `season` — все сыгранные матчи текущего сезона, если их ≥ N, иначе добираем до N последними матчами прошлого сезона.

**Учёт уже идущих продаж** (h = `days_to_match`):
- `A`: `max(уровень, sold_as_of)`, то есть только нижняя граница. `postprocess` и так её применяет, так что A — чистый эффект окна;
- `B`: `sold_as_of + (1 − share_z(h)) · уровень`, где `share_z(h)` — средняя доля итога зоны z, которую train-матчи фолда продали раньше чем за h дней до игры. Доля считается по `load_daily_sales(as_of)` только по train-матчам, утечки нет.

Сетка: окно ∈ {last, season} × N ∈ {5, 10, 15} × {A, B}. Выбор только по общему RMSE на валидации, тест не используется.

In [11]:
import numpy as np

from src.data import load_daily_sales


def zone_level(train, val, n, window):
    """Среднее tickets_total по зоне за окно матчей до as_of; для каждой строки val."""
    played = train.drop_duplicates("match_id").sort_values(["match_date", "match_id"])
    out = pd.Series(np.nan, index=val.index)
    for season, idx in val.groupby("season").groups.items():
        if window == "last":
            ids = played["match_id"].tail(n)
        else:  # весь текущий сезон, если в нём >= n матчей, иначе добираем из прошлых
            cur = played[played["season"] == season]
            past = played[played["season"] != season]
            ids = pd.concat([past["match_id"].tail(max(n - len(cur), 0)), cur["match_id"]])
        level = train[train["match_id"].isin(ids)].groupby("zone")["tickets_total"].mean()
        out[idx] = val.loc[idx, "zone"].map(level)
    return out.to_numpy()


def share_sold(train, val, as_of):
    """Средняя по train-матчам доля итога зоны, проданная раньше чем за days_to_match дней до игры."""
    daily = load_daily_sales(as_of)
    daily = daily[daily["match_id"].isin(train["match_id"])]
    total = train.set_index(["match_id", "zone"])["tickets_total"]
    total = total[total > 0]
    share = pd.Series(0.0, index=val.index)
    for h, idx in val.groupby("days_to_match").groups.items():
        early = daily[daily["days_before"] > h].groupby(["match_id", "zone"])["tickets"].sum()
        frac = (early.reindex(total.index, fill_value=0) / total).groupby(level="zone").mean()
        share[idx] = val.loc[idx, "zone"].map(frac).fillna(0)
    return share.to_numpy()


def make_smart(n, window, variant):
    def predict(train, val, as_of):
        level = zone_level(train, val, n, window)
        if variant == "A":
            return np.maximum(level, val["sold_as_of"])
        return val["sold_as_of"].to_numpy() + (1 - share_sold(train, val, as_of)) * level
    return predict

## Сетка: окно × N × вариант

In [12]:
grid = {}
for window in ["last", "season"]:
    for n in [5, 10, 15]:
        for variant in ["A", "B"]:
            grid[(window, n, variant)] = evaluate(make_smart(n, window, variant))

grid_table = pd.DataFrame([
    {"window": w, "N": n, "variant": v, "rmse": r.overall,
     **r.by_fold["rmse"].drop("all").to_dict(),
     "rmse_in_sale": r.by_status.loc["в продаже", "rmse"]}
    for (w, n, v), r in grid.items()
]).sort_values("rmse").reset_index(drop=True)
grid_table

,window,N,variant,rmse,2023-12-15,2024-11-25,2025-09-20,rmse_in_sale
0,season,10,A,135.2,172.2,79.1,148.5,110.9
1,season,15,A,136.8,172.2,79.1,153.8,110.5
2,last,15,A,137.7,175.7,76.6,153.8,112.8
3,last,10,A,143.4,192.7,76.6,148.5,118.1
4,season,5,A,147.9,172.2,79.1,187.0,124.8
5,season,10,B,148.5,196.5,87.0,152.6,149.2
6,season,15,B,150.0,196.5,87.0,157.5,149.5
7,last,15,B,151.1,200.1,85.0,157.5,152.3
8,last,10,B,157.0,216.9,84.8,152.6,157.9
9,season,5,B,159.6,196.5,87.0,188.4,154.8


In [13]:
best_key = min(grid, key=lambda k: grid[k].overall)
smart = grid[best_key]
best_window, best_n, best_variant = best_key
print(f"Лучшее: window={best_window}, N={best_n}, variant={best_variant}, RMSE={smart.overall:.1f}")

Лучшее: window=season, N=10, variant=A, RMSE=135.2


**Оговорка.** N, окно и вариант выбраны по тем же трём фолдам, на которых считается RMSE ниже. Поэтому оценка умного baseline слегка оптимистична: из 12 комбинаций выбрана лучшая. Для честного сравнения моделей команды стоит фиксировать этот выбор, а не подбирать заново.

## Сравнение с baseline

In [14]:
def compare(a, b, name_a="baseline", name_b="smart"):
    out = pd.concat({name_a: a, name_b: b}, axis=1)
    out["delta"] = out[name_b] - out[name_a]
    return out

print("По фолдам")
display(compare(res.by_fold["rmse"], smart.by_fold["rmse"]))
print("По зонам (все фолды)")
display(compare(res.by_zone["all"], smart.by_zone["all"]))
print("По статусу продаж")
display(compare(res.by_status["rmse"], smart.by_status["rmse"]))

По фолдам


,baseline,smart,delta
fold,,,
2023-12-15,172.2,172.2,0.0
2024-11-25,103.0,79.1,-24.0
2025-09-20,147.2,148.5,1.4
all,141.0,135.2,-5.8


По зонам (все фолды)


,baseline,smart,delta
zone,,,
1,41.6,36.8,-4.9
2,38.4,28.9,-9.5
3,63.3,59.1,-4.2
4,164.3,139.0,-25.3
5,109.4,119.5,10.1
6,257.0,225.7,-31.3
7,208.5,228.4,19.9
8,58.8,62.2,3.4


По статусу продаж


,baseline,smart,delta
in_sale,,,
ещё не в продаже,149.5,144.5,-5.0
в продаже,119.3,110.9,-8.4


In [15]:
smart.by_zone

fold,2023-12-15,2024-11-25,2025-09-20,all
zone,,,,
1,46.4,32.4,29.1,36.8
2,40.4,22.4,19.2,28.9
3,63.6,56.7,56.9,59.1
4,220.1,68.5,78.0,139.0
5,130.1,127.8,91.1,119.5
6,286.4,47.1,289.4,225.7
7,282.8,132.3,262.3,228.4
8,44.7,67.2,71.5,62.2


In [16]:
display(Markdown(f"# Smart baseline RMSE = {smart.overall:.1f} (baseline {res.overall:.1f})"))

# Smart baseline RMSE = 135.2 (baseline 141.0)

**Выводы по умному baseline:**
- Лучшая комбинация: окно `season`, N=10, вариант A. RMSE 135.2 против 141.0, это −5.8 (−4%).
- **Весь выигрыш даёт один фолд.** На 2024-11-25 улучшение −24. На 2023-12-15 изменений нет: в train только один сезон, поэтому «текущий сезон» совпадает со «всеми матчами». На 2025-09-20 немного хуже (+1.4): в сезоне сыграно всего 5 матчей, и окно добирает конец прошлого сезона. Улучшение реальное, но хрупкое.
- У `season` при N=10 и N=15 первые два фолда совпадают, так что N выбран фактически по одному фолду 2025-09-20. **На test N ничего не меняет:** в 2025/26 уже сыграно 17 матчей ≥ N, и уровень зоны равен среднему по всем 17 матчам сезона.
- По зонам: заметный выигрыш в 4 и 6 (−25, −31) и в 1–3, но зоны 5 и 7 стали хуже (+10, +20). Дрейф по сезонам в разных зонах идёт в разные стороны.
- **Вариант B хуже A при любом окне и N**, в том числе на матчах в продаже (149 против 111). Доля проданного к h дням до игры сильно зависит от даты открытия продаж и ажиотажа. Например, M023 и M024 (декабрь 2023) продали 87–92% итога за 9–19 дней до игры, а кривая ожидала 19–45%, поэтому B добавляет полный остаток и сильно завышает. Для модели стоит пробовать долю от прошедшей части окна продаж (`days_since_open / (days_since_open + days_to_match)`) или темп `sold_last_7d`, а не фиксированную кривую по h.

## Прогноз умного baseline на test

In [17]:
if smart.overall < res.overall:
    smart_sub = predict_test(make_smart(best_n, best_window, best_variant))
    assert len(smart_sub) == 136 and (smart_sub["tickets_total"] >= 0).all()
    smart_sub.to_csv(out / "smart_baseline.csv", index=False, encoding="utf-8")
    print(f"Сохранено: {out / 'smart_baseline.csv'}")
    display(smart_sub.pivot(index="match_id", columns="zone", values="tickets_total"))
else:
    print("Умный baseline не лучше baseline — файл не сохраняем")

Сохранено: ..\submissions\smart_baseline.csv


zone,1,2,3,4,5,6,7,8
match_id,,,,,,,,
M086,453,265,588,871,761,649,785,425
M087,453,265,588,871,761,700,785,425
M088,453,265,588,871,761,649,785,425
M089,453,265,588,871,761,649,785,425
M090,453,265,588,871,761,649,785,425
M091,453,265,588,871,761,649,785,425
M092,453,265,588,871,761,649,785,425
M093,453,265,588,871,761,649,785,425
M094,453,265,588,871,761,649,785,425
